# 00 — Scope & Environment

Notebook này xác định phạm vi bài toán, môi trường thực thi,
đặc điểm dữ liệu đầu vào và các giới hạn tài nguyên trước khi
triển khai pipeline Machine Learning.

Mục tiêu:
- Xác định rõ bài toán và target.
- Ghi nhận môi trường thực thi.
- Khảo sát kích thước và schema của raw datasets.
- Đánh giá khả năng xử lý dữ liệu theo tài nguyên hiện có.
- Đưa ra các quyết định kỹ thuật ban đầu cho pipeline.

## 1. Project Scope

Xác định phạm vi bài toán trước khi thực hiện EDA và
feature engineering.

In [2]:
PROJECT_NAME = "Machine Learning Final Project"
DATA_DIR = "../data"
RAW_DATA_DIR = "../data/raw"

TARGET_COLUMN = "unit_sales"

TASK_TYPE = "Time Series Forecasting"

print(f"Project: {PROJECT_NAME}")
print(f"Task: {TASK_TYPE}")
print(f"Target: {TARGET_COLUMN}")
print(f"Raw data directory: {RAW_DATA_DIR}")

Project: Machine Learning Final Project
Task: Time Series Forecasting
Target: unit_sales
Raw data directory: ../data/raw


### Scope Notes

- Bài toán được xác định là dự báo doanh số.
- Target ban đầu dự kiến là `unit_sales`.
- Raw datasets được lưu tại `data/raw/`.
- Các quyết định về feature và validation sẽ được xác định
  sau khi hoàn thành data validation.

  -> Chưa chốt nếu thông tin chưa chắc chắn

## 2. Runtime Environment

Ghi nhận môi trường phần cứng và phần mềm hiện tại để xác định
các giới hạn tài nguyên có thể ảnh hưởng đến quá trình xử lý dữ liệu
và huấn luyện mô hình.

In [3]:
import os
import platform
import sys

print("Operating System:", platform.platform())
print("Python Version:", sys.version)
print("Python Executable:", sys.executable)

Operating System: Linux-7.0.0-30-generic-x86_64-with-glibc2.39
Python Version: 3.11.16 (main, Aug 27 2026, 14:44:21) [GCC 14.3.0]
Python Executable: /home/phat/miniconda3/envs/grocery_env/bin/python


### Environment Notes

Kết quả trên được sử dụng để ghi nhận môi trường thực thi của project.

In [4]:
import os
import platform

print("CPU:", platform.processor())
print("Logical CPU cores:", os.cpu_count())

CPU: x86_64
Logical CPU cores: 8


### CPU Notes

Ghi nhận số logical CPU cores để làm cơ sở cân nhắc mức độ
parallelism trong các bước preprocessing và training.

In [5]:
import psutil

memory = psutil.virtual_memory()

print(f"Total RAM: {memory.total / (1024**3):.2f} GB")
print(f"Available RAM: {memory.available / (1024**3):.2f} GB")
print(f"Used RAM: {memory.used / (1024**3):.2f} GB")
print(f"RAM Usage: {memory.percent:.1f}%")

Total RAM: 7.50 GB
Available RAM: 2.40 GB
Used RAM: 5.10 GB
RAM Usage: 68.0%


### RAM Notes

RAM khả dụng là một trong những yếu tố chính quyết định cách
đọc và xử lý các dataset lớn.

Kết quả sẽ được sử dụng ở phần Resource Assessment để quyết định
có cần chunking, dtype optimization hoặc các chiến lược giảm
memory usage hay không.

In [6]:
import shutil
import subprocess

gpu_available = shutil.which("nvidia-smi") is not None

if gpu_available:
    result = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        capture_output=True,
        text=True
    )
    print(result.stdout.strip())
else:
    print("NVIDIA GPU: Not detected")

NVIDIA GPU: Not detected


### GPU Notes

Nếu không có GPU, việc lựa chọn mô hình và cấu hình training
sẽ ưu tiên các phương pháp có khả năng chạy hiệu quả trên CPU.

Nếu có GPU, thông tin VRAM sẽ được sử dụng để đánh giá khả năng
huấn luyện các mô hình có hỗ trợ GPU.

In [7]:
import shutil

disk = shutil.disk_usage(".")

print(f"Total disk: {disk.total / (1024**3):.2f} GB")
print(f"Used disk: {disk.used / (1024**3):.2f} GB")
print(f"Free disk: {disk.free / (1024**3):.2f} GB")

Total disk: 233.18 GB
Used disk: 119.54 GB
Free disk: 101.73 GB


## 3. Dataset Inventory

Khảo sát toàn bộ dữ liệu trong `data/raw/` mà không thay đổi
nội dung của raw datasets.

Các thông tin cần thu thập:

- File name
- File size
- Number of rows
- Number of columns
- Column names
- Data types

In [8]:
from pathlib import Path

RAW_DATA_DIR = Path("../data/raw")

csv_files = sorted(RAW_DATA_DIR.glob("*.csv"))

print(f"Found {len(csv_files)} CSV files:\n")

for path in csv_files:
    print(path.name)

Found 8 CSV files:

holidays_events.csv
items.csv
oil.csv
sample_submission.csv
stores.csv
test.csv
train.csv
transactions.csv


In [9]:
import pandas as pd

file_inventory = []

for path in csv_files:
    file_inventory.append({
        "file": path.name,
        "size_mb": path.stat().st_size / (1024**2)
    })

file_inventory_df = (
    pd.DataFrame(file_inventory)
    .sort_values("size_mb", ascending=False)
    .reset_index(drop=True)
)

file_inventory_df

,file,size_mb
0,train.csv,4765.941895
1,test.csv,120.318438
2,sample_submission.csv,38.571913
3,transactions.csv,1.480710
4,items.csv,0.097123
5,holidays_events.csv,0.021276
6,oil.csv,0.019627
7,stores.csv,0.001323


### File Size Notes

Bảng trên cho phép xác định các raw datasets có kích thước lớn
nhất và là cơ sở ban đầu để đánh giá nguy cơ memory bottleneck.

In [10]:
dataset_summary = []

for path in csv_files:
    header = pd.read_csv(path, nrows=0)

    with open(path, "rb") as f:
        row_count = sum(1 for _ in f) - 1

    dataset_summary.append({
        "file": path.name,
        "rows": row_count,
        "columns": len(header.columns),
    })

dataset_summary_df = (
    pd.DataFrame(dataset_summary)
    .sort_values("rows", ascending=False)
    .reset_index(drop=True)
)

dataset_summary_df

,file,rows,columns
0,train.csv,125497040,6
1,sample_submission.csv,3370464,2
2,test.csv,3370464,5
3,transactions.csv,83488,3
4,items.csv,4100,4
5,oil.csv,1218,2
6,holidays_events.csv,350,6
7,stores.csv,54,5


In [11]:
schema_summary = {}

for path in csv_files:
    df_sample = pd.read_csv(path, nrows=1000)

    schema_summary[path.name] = pd.DataFrame({
        "column": df_sample.columns,
        "dtype": df_sample.dtypes.astype(str).values,
        "sample_memory_mb": [
            df_sample[[col]].memory_usage(deep=True).sum() / (1024**2)
            for col in df_sample.columns
        ]
    })

for filename, schema in schema_summary.items():
    print(f"\n{'=' * 60}")
    print(filename)
    display(schema)


holidays_events.csv


,column,dtype,sample_memory_mb
0,date,str,0.006134
1,type,str,0.005183
2,locale,str,0.005032
3,locale_name,str,0.005339
4,description,str,0.009858
5,transferred,bool,0.000460



items.csv


,column,dtype,sample_memory_mb
0,item_nbr,int64,0.007755
1,family,str,0.015719
2,class,int64,0.007755
3,perishable,int64,0.007755



oil.csv


...

,column,dtype,sample_memory_mb
0,id,int64,0.007755
1,date,str,0.017292
2,store_nbr,int64,0.007755
3,item_nbr,int64,0.007755
4,onpromotion,bool,0.001080



train.csv


,column,dtype,sample_memory_mb
0,id,int64,0.007755
1,date,str,0.017292
2,store_nbr,int64,0.007755
3,item_nbr,int64,0.007755
4,unit_sales,float64,0.007755
5,onpromotion,float64,0.007755



transactions.csv


,column,dtype,sample_memory_mb
0,date,str,0.017292
1,store_nbr,int64,0.007755
2,transactions,int64,0.007755


### Dataset Inventory Notes

Từ inventory, xác định:

1. Dataset nào lớn nhất? -> train.csv
2. Dataset nào có số dòng lớn nhất? -> train.csv
3. Có column nào có dtype không tối ưu không? -> chưa chốt
4. Có column categorical/string cần xem xét không? -> có 
5. Có dataset nào cần xử lý theo chunk không? -> train.csv có 4 GB trở lên

Các kết luận chi tiết về missing values, duplicates và data quality
sẽ được thực hiện trong notebook Data Validation tiếp theo.

## 4. Resource Assessment

Đánh giá mối quan hệ giữa kích thước dữ liệu và tài nguyên
phần cứng hiện có.

Mục tiêu không phải tối ưu hóa ngay, mà xác định các
rủi ro tài nguyên trước khi xây dựng pipeline.

In [12]:
file_inventory_df.sort_values(
    "size_mb",
    ascending=False
).head(10)

,file,size_mb
0,train.csv,4765.941895
1,test.csv,120.318438
2,sample_submission.csv,38.571913
3,transactions.csv,1.480710
4,items.csv,0.097123
5,holidays_events.csv,0.021276
6,oil.csv,0.019627
7,stores.csv,0.001323


In [13]:
memory_estimates = []

for path in csv_files:
    sample = pd.read_csv(path, nrows=10_000)

    sample_memory = sample.memory_usage(deep=True).sum()
    bytes_per_row = sample_memory / len(sample)

    row_count = dataset_summary_df.loc[
        dataset_summary_df["file"] == path.name,
        "rows"
    ].iloc[0]

    estimated_memory = bytes_per_row * row_count

    memory_estimates.append({
        "file": path.name,
        "estimated_dataframe_memory_gb":
            estimated_memory / (1024**3)
    })

memory_estimates_df = (
    pd.DataFrame(memory_estimates)
    .sort_values(
        "estimated_dataframe_memory_gb",
        ascending=False
    )
    .reset_index(drop=True)
)

memory_estimates_df

,file,estimated_dataframe_memory_gb
0,train.csv,6.780480
1,test.csv,0.135018
2,sample_submission.csv,0.050265
3,transactions.csv,0.002645
4,items.csv,0.000157
5,holidays_events.csv,0.000031
6,oil.csv,0.000030
7,stores.csv,0.000003


### Memory Assessment

Estimated DataFrame memory được ước tính từ một sample dữ liệu
và có thể khác với memory usage thực tế.

So sánh:

- Available RAM
- Estimated DataFrame memory
- Number of large datasets cần xử lý đồng thời

để xác định nguy cơ Out-of-Memory.

In [14]:
available_ram_gb = psutil.virtual_memory().available / (1024**3)

largest_estimated_memory_gb = (
    memory_estimates_df[
        "estimated_dataframe_memory_gb"
    ].max()
)

print(f"Available RAM: {available_ram_gb:.2f} GB")
print(
    f"Largest estimated DataFrame memory: "
    f"{largest_estimated_memory_gb:.2f} GB"
)

Available RAM: 2.66 GB
Largest estimated DataFrame memory: 6.78 GB


### Potential Bottlenecks

Dựa trên kết quả phía trên, đánh giá:

- Có dataset nào có thể chiếm phần lớn RAM không?
-> tập train.csv
- Có cần tránh load nhiều dataset lớn đồng thời không?
-> Cần để tránh tràn RAM và quá tải bộ nhớ
- Có cần đọc dữ liệu theo chunks không?
-> Chunk processing là bắt buộc đối với các thao tác đọc toàn bộ train.csv; kích thước chunk sẽ được benchmark ở bước preprocessing.
- Có cần tối ưu dtype không?
-> không
- Có cần giảm số lượng intermediate copies không?
-> Cần

## 5. Initial Engineering Decisions

Tổng hợp các kết quả thu được từ Scope, Runtime Environment,
Dataset Inventory và Resource Assessment để đưa ra các quyết định
kỹ thuật ban đầu.

Các quyết định ở đây là provisional và có thể được điều chỉnh
sau Data Validation, EDA và benchmarking.

In [15]:
environment_summary = {
    "python": platform.python_version(),
    "cpu_cores": os.cpu_count(),
    "ram_total_gb": psutil.virtual_memory().total / (1024**3),
    "ram_available_gb": psutil.virtual_memory().available / (1024**3),
    "disk_free_gb": shutil.disk_usage(".").free / (1024**3),
    "num_raw_csv": len(csv_files),
}

environment_summary

{'python': '3.11.16',
 'cpu_cores': 8,
 'ram_total_gb': 7.500282287597656,
 'ram_available_gb': 2.661052703857422,
 'disk_free_gb': 101.72613525390625,
 'num_raw_csv': 8}

### Initial Engineering Decisions

#### Data Loading
- Không mặc định load toàn bộ raw datasets vào RAM cùng lúc.
- Dataset lớn sẽ được benchmark trước khi quyết định chiến lược loading.
-> RAM tối đa có thể sử dụng giao động từ 1,5GB đến 3GB tùy vào tác vụ đang sử dụng trên máy tính.

#### Memory
- Xem xét dtype optimization cho numerical và categorical columns.
- Sử dụng chunk processing nếu dataset hoặc intermediate dataset
  vượt giới hạn tài nguyên thực tế.
-> Phải sử dụng chunk proccessing vì các tập dataset như train.csv, transactions.csv, test.csv, sample_submission.csv thường nhiều hơn 1 GB.

#### Validation
- Vì bài toán có yếu tố thời gian, validation strategy sẽ được
  thiết kế theo temporal ordering thay vì random split.
-> tạm thời đồng ý với kết luận này.

#### Feature Engineering
- Feature engineering sẽ được thực hiện sau khi hoàn thành
  data validation và xác định rõ nguy cơ data leakage.
-> Sẽ cập nhật sau.

#### Modeling
- Model selection sẽ phụ thuộc vào kích thước dữ liệu,
  thời gian training và tài nguyên phần cứng.
-> Sẽ cập nhật sau.

-> Cần cập nhật khi có số liệu thực tế

## Conclusion

Notebook này thiết lập baseline về:

1. Project scope
2. Runtime environment
3. Dataset characteristics
4. Resource constraints
5. Initial engineering decisions

Notebook tiếp theo sẽ sử dụng các kết quả này để thực hiện
Data Understanding & Data Validation trên raw datasets.

## 4. Resource Assessment
    ├── Current RAM
    ├── Estimated full DataFrame memory
    ├── Chunk-size benchmark
    │   ├── 50k
    │   ├── 100k
    │   ├── 250k
    │   ├── 500k
    │   └── 1M
    └── Recommended processing strategy